# Phase 11b Pilot — HumanEval (Code Execution)

**Goal**: Verify that spectral entropy features extracted from a code-generation model's token logits
can distinguish passing from failing HumanEval solutions. This is a **GO/NO-GO pilot** — N=20,
one model. If all gates pass we proceed to a full Phase 11b run (N=164, multi-model).

## Gates
| ID | Gate | Threshold |
|----|------|-----------|
| G0 | At least 5 problems solved (both classes present) | ≥ 5 passed |
| G1 | Entropy traces non-degenerate | std > 0 on ≥ 90% of traces |
| G2 | `extract_all_features` returns non-NaN | ≥ 15/20 problems |
| G3 | No Colab crash during `execute_python_solution` | subprocess exits cleanly |

**Verdict**: All 4 gates pass → **GO** for Phase 11b full HumanEval run.

In [ ]:
# ── Cell 2: Setup ─────────────────────────────────────────────────────────────
import os, sys, shutil

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['HF_HOME'] = '/content/drive/MyDrive/hf_cache'

from google.colab import drive
drive.mount('/content/drive')

REPO_DIR = '/content/hallucination_detection'
if os.path.exists(REPO_DIR) and not os.path.exists(os.path.join(REPO_DIR, 'spectral_utils')):
    shutil.rmtree(REPO_DIR)
if not os.path.exists(REPO_DIR):
    os.system(f'git clone -b master https://github.com/omrisegev/hallucination_detection.git {REPO_DIR}')
else:
    os.system(f'git -C {REPO_DIR} pull -q')
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

os.system('pip install -q "transformers>=4.40" accelerate datasets bitsandbytes autoawq scipy')

from spectral_utils import (
    load_model, generate_full, free_memory,
    extract_all_features, sw_var_peak_adaptive, FEAT_NAMES,
    load_cache, save_cache,
    zscore, boot_auc, nadler_fuse, best_nadler_on,
    load_humaneval, humaneval_prompt, is_correct_humaneval,
    execute_python_solution, run_humaneval_episode,
)
import datasets  # freeze pyarrow C extensions before any gptqmodel install
print('spectral_utils imported OK')
print(f'FEAT_NAMES ({len(FEAT_NAMES)}):', FEAT_NAMES)

In [ ]:
# ── Cell 3: Config ────────────────────────────────────────────────────────────
MODEL_KEY   = 'qwen25_7b'
MODEL_ID    = 'Qwen/Qwen2.5-7B-Instruct'
N_SAMPLES   = 20
MAX_ATTEMPTS = 3
MAX_NEW      = 512
TEMP         = 1.0

CACHE_DIR  = '/content/drive/MyDrive/hallucination_detection/cache/phase11b_humaneval_pilot'
RAW_DIR    = os.path.join(CACHE_DIR, 'raw')
FEAT_DIR   = os.path.join(CACHE_DIR, 'features')
RES_DIR    = os.path.join(CACHE_DIR, 'results')
for d in [RAW_DIR, FEAT_DIR, RES_DIR]:
    os.makedirs(d, exist_ok=True)

CELL_KEY = f'{MODEL_KEY}_n{N_SAMPLES}'
print(f'Config: {MODEL_KEY} | N={N_SAMPLES} | attempts={MAX_ATTEMPTS} | T={TEMP}')
print(f'Cache: {CACHE_DIR}')

In [ ]:
# ── Cell 4: Load dataset ───────────────────────────────────────────────────────
problems = load_humaneval(N_SAMPLES)
print(f'\nSample problem:')
print(f"  task_id:     {problems[0]['task_id']}")
print(f"  entry_point: {problems[0]['entry_point']}")
print(f"  prompt:\n{problems[0]['prompt'][:300]}")
print(f"  test (first 100 chars): {problems[0]['test'][:100]}")

In [ ]:
# ── Cell 5: Load model ────────────────────────────────────────────────────────
mdl, tok = load_model(MODEL_ID)
print('Model loaded.')

In [ ]:
# ── Cell 6: Inference ─────────────────────────────────────────────────────────
import pickle, time

RAW_PATH     = os.path.join(RAW_DIR, f'{CELL_KEY}_trajs.pkl')
CKPT_PATH    = os.path.join(RAW_DIR, f'{CELL_KEY}_ckpt.pkl')
CKPT_EVERY   = 5
FORCE_RERUN  = False

if not FORCE_RERUN and os.path.exists(RAW_PATH):
    with open(RAW_PATH, 'rb') as f:
        trajs = pickle.load(f)
    print(f'Loaded {len(trajs)} trajectories from {RAW_PATH}')
else:
    # Resume from checkpoint if available
    if os.path.exists(CKPT_PATH) and not FORCE_RERUN:
        with open(CKPT_PATH, 'rb') as f:
            trajs = pickle.load(f)
        print(f'Resuming from checkpoint: {len(trajs)}/{N_SAMPLES} done')
    else:
        trajs = []

    start_idx = len(trajs)
    t0 = time.time()

    for i in range(start_idx, N_SAMPLES):
        row = problems[i]
        traj = run_humaneval_episode(
            mdl, tok, row,
            T=TEMP, max_attempts=MAX_ATTEMPTS, max_new=MAX_NEW,
        )
        trajs.append(traj)

        elapsed = time.time() - t0
        avg = elapsed / (i - start_idx + 1)
        print(f'[{i+1:3d}/{N_SAMPLES}] {row["task_id"]:30s} | '
              f'passed={traj["any_passed"]} attempts={traj["n_attempts"]} | '
              f'~{avg*(N_SAMPLES-i-1)/60:.1f}min left')

        if (i + 1) % CKPT_EVERY == 0:
            with open(CKPT_PATH, 'wb') as f:
                pickle.dump(trajs, f)
            print(f'  → checkpoint saved ({len(trajs)} trajs)')

    with open(RAW_PATH, 'wb') as f:
        pickle.dump(trajs, f)
    print(f'Done. Saved {len(trajs)} trajs to {RAW_PATH}')

n_passed = sum(t['any_passed'] for t in trajs)
print(f'\nPass@attempts: {n_passed}/{len(trajs)} ({100*n_passed/len(trajs):.1f}%)')

In [ ]:
# ── Cell 7: Unload model ──────────────────────────────────────────────────────
del mdl, tok
free_memory()
print('Model unloaded.')

In [ ]:
# ── Cell 8: Feature extraction ────────────────────────────────────────────────
# For each trajectory we use the FIRST attempt's entropy trace as the signal.
# Label: any_passed (solved within MAX_ATTEMPTS).
import pickle
import numpy as np
from spectral_utils import extract_all_features, sw_var_peak_adaptive

FEAT_PATH = os.path.join(FEAT_DIR, f'{CELL_KEY}_feats.pkl')
FORCE_RECOMPUTE = False

if not FORCE_RECOMPUTE and os.path.exists(FEAT_PATH):
    with open(FEAT_PATH, 'rb') as f:
        feat_data = pickle.load(f)
    print(f'Loaded features from {FEAT_PATH}')
else:
    feat_data = {'features': [], 'labels': [], 'task_ids': [], 'n_attempts': []}
    n_nan = 0

    for traj in trajs:
        label = int(traj['any_passed'])
        # Use first attempt entropy trace (measures model's uncertainty at first shot)
        ents = traj['steps'][0]['token_entropies']

        if len(ents) < 4:
            n_nan += 1
            continue

        feats = extract_all_features(ents)
        feats['sw_var_peak'] = sw_var_peak_adaptive(ents)  # adaptive override

        feat_data['features'].append(feats)
        feat_data['labels'].append(label)
        feat_data['task_ids'].append(traj['task_id'])
        feat_data['n_attempts'].append(traj['n_attempts'])

    with open(FEAT_PATH, 'wb') as f:
        pickle.dump(feat_data, f)
    print(f'Extracted features for {len(feat_data["labels"])} problems ({n_nan} skipped, too short)')

labels   = np.array(feat_data['labels'])
n_pos    = labels.sum()
n_neg    = len(labels) - n_pos
print(f'Label balance: {n_pos} passed / {n_neg} failed (total {len(labels)})')

# G1 check: entropy trace degeneracy
all_ents = [traj['steps'][0]['token_entropies'] for traj in trajs if len(traj['steps'][0]['token_entropies']) >= 4]
non_degen = sum(1 for e in all_ents if np.std(e) > 0)
print(f'G1 check: {non_degen}/{len(all_ents)} traces have std > 0')

In [ ]:
# ── Cell 9: GO/NO-GO Gate Cell ────────────────────────────────────────────────
import numpy as np

n_total   = len(trajs)
n_valid   = len(feat_data['labels'])
labels_np = np.array(feat_data['labels'])
n_passed_gate = int(labels_np.sum())

# G0: class balance
g0 = n_passed_gate >= 5
# G1: non-degenerate traces (computed in Cell 8)
g1 = non_degen / max(len(all_ents), 1) >= 0.90
# G2: feature extraction coverage
g2 = n_valid >= 15
# G3: no subprocess crash (if we got here without exception, we're clean)
g3 = True

print("=" * 50)
print("  Phase 11b HumanEval Pilot — GO/NO-GO Report")
print("=" * 50)
print(f"  G0  Class balance (≥5 passed):   {'PASS ✓' if g0 else 'FAIL ✗'}  ({n_passed_gate}/{n_total} passed)")
print(f"  G1  Non-degenerate traces (≥90%): {'PASS ✓' if g1 else 'FAIL ✗'}  ({non_degen}/{len(all_ents)} = {100*non_degen/max(len(all_ents),1):.0f}%)")
print(f"  G2  Feature coverage (≥15/20):   {'PASS ✓' if g2 else 'FAIL ✗'}  ({n_valid}/{n_total} valid)")
print(f"  G3  No subprocess crash:          {'PASS ✓' if g3 else 'FAIL ✗'}")
print("=" * 50)
verdict = "GO  → proceed to Phase 11b full run" if (g0 and g1 and g2 and g3) \
          else "NO-GO → investigate failing gates"
print(f"  VERDICT: {verdict}")
print("=" * 50)

In [ ]:
# ── Cell 10: Indicative AUC table ─────────────────────────────────────────────
# N=20 — CIs will be wide, treat as directional signal only.
import numpy as np
from spectral_utils import boot_auc, FEAT_NAMES

feats_list = feat_data['features']
labels_arr = np.array(feat_data['labels'])

if len(np.unique(labels_arr)) < 2:
    print('WARNING: only one class present — skip AUC (G0 failed)')
else:
    rows = []
    for feat_name in FEAT_NAMES + ['sw_var_peak']:
        vals = np.array([f.get(feat_name, float('nan')) for f in feats_list])
        mask = ~np.isnan(vals)
        if mask.sum() < 4 or len(np.unique(labels_arr[mask])) < 2:
            continue
        auc, lo, hi = boot_auc(vals[mask], labels_arr[mask])
        # orient: higher feature → more likely to pass
        if auc < 0.5:
            auc, lo, hi = 1 - auc, 1 - hi, 1 - lo
        rows.append((feat_name, auc, lo, hi))

    rows.sort(key=lambda x: -x[1])
    print(f"\n{'Feature':<28} {'AUC':>6}  {'95% CI':>13}")
    print("-" * 52)
    for feat_name, auc, lo, hi in rows:
        print(f"  {feat_name:<26} {auc:.3f}  [{lo:.3f}, {hi:.3f}]")

    # Quick Nadler check (N=20 is small — just verify it runs)
    from spectral_utils import best_nadler_on
    feat_matrix_rows = []
    valid_labels     = []
    for feats, lbl in zip(feats_list, labels_arr):
        row = [feats.get(k, float('nan')) for k in FEAT_NAMES if k != 'trace_length' and k != 'pe_min']
        if not any(np.isnan(row)):
            feat_matrix_rows.append(row)
            valid_labels.append(lbl)

    feat_arr_all = np.array(feat_matrix_rows)
    keys_nadler  = [k for k in FEAT_NAMES if k not in ('trace_length', 'pe_min')]

    if len(np.unique(valid_labels)) >= 2 and len(valid_labels) >= 8:
        nadler_auc, lo, hi, subset, _ = best_nadler_on(feat_arr_all, np.array(valid_labels),
                                                         keys_nadler, normalize=True)
        print(f"\nNadler (indicative, N={len(valid_labels)}): {nadler_auc:.3f} [{lo:.3f}, {hi:.3f}]  subset={subset}")
    else:
        print('\nNadler: insufficient samples or single class — skip')